# Q&A Pairs: BigQuery → Weaviate

This notebook loads curated Q&A pairs from customer support conversations in BigQuery and uploads them to Weaviate.

**Data Source:**
- BigQuery table: `kittl-data-platform.bl.intercom_qa_pairs_ob_mascot_TEST`
- Filter: `quality_rating = "Good"`
- Each Q&A pair is uploaded as a single chunk (not split)

**Weaviate Collection:** `MascotQAPairs`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path("/workspaces/py/projects/onboarding_mascot")
sys.path.insert(0, str(project_root))

# Load environment variables from root .env
from dotenv import load_dotenv
load_dotenv(dotenv_path="/workspaces/py/.env", override=True)

print("✅ Environment loaded")

✅ Environment loaded


## 2. Query BigQuery for Q&A Pairs

In [13]:
from google.cloud import bigquery
import os

# Initialize BigQuery client
credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

if credentials_path:
    bq_client = bigquery.Client.from_service_account_json(credentials_path)
else:
    bq_client = bigquery.Client(project=project_id)

print(f"✅ Connected to BigQuery (project: {project_id})")

✅ Connected to BigQuery (project: kittl-data-platform)


In [14]:
# Query for good Q&A pairs
query = """
SELECT qa_pair 
FROM `kittl-data-platform.prod_ol.intercom_resolved_conversations_mascot_pairs` 
WHERE quality_rating = "Good"
"""

print("🔍 Querying BigQuery...")
results = bq_client.query(query).to_dataframe()

print(f"\n✅ Retrieved {len(results)} Q&A pairs")
print(f"\n📊 Sample data:")
print(results.head())

🔍 Querying BigQuery...


/workspaces/py/dist/export/python/virtualenvs/python_default/3.10.18/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



✅ Retrieved 368 Q&A pairs

📊 Sample data:
                                             qa_pair
0  User Question: How can I check my AI tokens or...
1  User Question: Why are my designs downloading ...
2  User Question: Why am I being asked to upgrade...
3  User Question: How are AI credits consumed in ...
4  User Question: How can I export my Kittl desig...


## 3. Preview Q&A Pairs

In [15]:
# Show first few Q&A pairs
print("📝 Example Q&A Pairs:\n")
print("="*80)

for i, row in results.head(3).iterrows():
    print(f"\nPair {i+1}:")
    print(row['qa_pair'])
    print("-"*80)

📝 Example Q&A Pairs:


Pair 1:
User Question: How can I check my AI tokens or credits on Kittl?
Answer: You can check your AI tokens in two ways:
1.  **Kittl Editor:** Go to the editor, look at the lower-left corner, and click the (*) icon located above the help button. This section also provides options to top-up or buy tokens.
2.  **Team Settings Page.**
--------------------------------------------------------------------------------

Pair 2:
User Question: Why are my designs downloading as empty white artboards, and how can I fix this?
Answer: Designs downloading as empty white artboards often occur when the object's layer is outside the Artboard. To fix this, go to the "Layers" panel and drag the missing design layer down so it is positioned under the artboard.
--------------------------------------------------------------------------------

Pair 3:
User Question: Why am I being asked to upgrade when trying to create an image on Kittl, despite having a plan, and how do AI tokens wo

## 3.5. Add Manual Q&A Pairs (Optional)

Add manually created Q&A pairs here to supplement the BigQuery data. These will be appended to the results before uploading to Weaviate.


In [16]:
# import pandas as pd

# # Add your manual Q&A pairs here
# # Follow the same format: "User Question: ... \nAnswer: ..."
# manual_qa_list = [
#     # "User Question: Can I buy more AI tokens if I'm on the Free Plan?\nAnswer: No, token bundles are only available for users on paid plans (Pro or Expert). Free Plan users need to upgrade to a paid plan first before they can purchase additional token bundles. Check out our pricing page at https://www.kittl.com/pricing to see which plan works best for you.",
    
#     # Add more Q&A pairs below (one string per pair):
#     # "User Question: Your question here?\nAnswer: Your answer here.",
# ]

# # Convert to DataFrame and append to results
# if manual_qa_list:
#     manual_df = pd.DataFrame({"qa_pair": manual_qa_list})
#     results = pd.concat([results, manual_df], ignore_index=True)
#     print(f"✅ Added {len(manual_qa_list)} manual Q&A pairs")
#     print(f"📊 Total Q&A pairs: {len(results)}")
# else:
#     print("ℹ️  No manual Q&A pairs to add")


## 4. Connect to Weaviate

In [17]:
import weaviate
import os

# Weaviate credentials
WEAVIATE_URL = os.getenv("WEAVIATE_URL")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY_DATA_BOT")

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
    headers={
        "X-OpenAI-Api-Key": OPENAI_API_KEY
    }
)

print("✅ Connected to Weaviate!")
print(f"Cluster: {WEAVIATE_URL}")

✅ Connected to Weaviate!
Cluster: cxhkpu4nslcn5arn5xlug.c0.europe-west3.gcp.weaviate.cloud


## 5. Create Weaviate Collection for Q&A Pairs

In [ ]:
from weaviate.classes.config import Configure, Property, DataType

collection_name = "MascotQAPairs"

# Check if collection exists and delete it
if client.collections.exists(collection_name):
    client.collections.delete(collection_name)
    print(f"🗑️  Deleted existing collection: {collection_name}")

# Create collection
collection = client.collections.create(
    name=collection_name,
    description="Q&A pairs from customer support conversations",
    
    vectorizer_config=Configure.Vectorizer.text2vec_openai(
        model="text-embedding-3-small",
        vectorize_collection_name=False
    ),
    inverted_index_config=Configure.inverted_index(
        bm25_b=0.7,
        bm25_k1=1.25,
        index_null_state=False,
        index_property_length=False,
        index_timestamps=False
    ),
    
    properties=[
        Property(
            name="qa_pair",
            data_type=DataType.TEXT,
            description="Complete Q&A pair (question + answer)",
            index_searchable=True,
            skip_vectorization=False
        ),
        Property(
            name="source",
            data_type=DataType.TEXT,
            description="Source of the Q&A pair",
            skip_vectorization=True
        )
    ]
)

print(f"✅ Collection '{collection_name}' created successfully!")

🗑️  Deleted existing collection: IntercomQAPairs


/workspaces/py/dist/export/python/virtualenvs/python_default/3.10.18/lib/python3.10/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


✅ Collection 'IntercomQAPairs' created successfully!


## 6. Upload Q&A Pairs to Weaviate

In [19]:
print(f"📦 Uploading {len(results)} Q&A pairs to Weaviate...\n")

# Get the collection
collection = client.collections.get(collection_name)

# Upload in batch
with collection.batch.dynamic() as batch:
    for i, row in results.iterrows():
        batch.add_object(
            properties={
                "qa_pair": row["qa_pair"],
                "source": "intercom_customer_support"
            }
        )
        
        if (i + 1) % 10 == 0:
            print(f"  Uploaded {i + 1}/{len(results)} Q&A pairs...")

print(f"\n✅ Successfully uploaded {len(results)} Q&A pairs!")

# Verify
total_objects = collection.aggregate.over_all(total_count=True)
print(f"📊 Total objects in collection: {total_objects.total_count}")

📦 Uploading 368 Q&A pairs to Weaviate...

  Uploaded 10/368 Q&A pairs...
  Uploaded 20/368 Q&A pairs...
  Uploaded 30/368 Q&A pairs...
  Uploaded 40/368 Q&A pairs...
  Uploaded 50/368 Q&A pairs...
  Uploaded 60/368 Q&A pairs...
  Uploaded 70/368 Q&A pairs...
  Uploaded 80/368 Q&A pairs...
  Uploaded 90/368 Q&A pairs...
  Uploaded 100/368 Q&A pairs...
  Uploaded 110/368 Q&A pairs...
  Uploaded 120/368 Q&A pairs...
  Uploaded 130/368 Q&A pairs...
  Uploaded 140/368 Q&A pairs...
  Uploaded 150/368 Q&A pairs...
  Uploaded 160/368 Q&A pairs...
  Uploaded 170/368 Q&A pairs...
  Uploaded 180/368 Q&A pairs...
  Uploaded 190/368 Q&A pairs...
  Uploaded 200/368 Q&A pairs...
  Uploaded 210/368 Q&A pairs...
  Uploaded 220/368 Q&A pairs...
  Uploaded 230/368 Q&A pairs...
  Uploaded 240/368 Q&A pairs...
  Uploaded 250/368 Q&A pairs...
  Uploaded 260/368 Q&A pairs...
  Uploaded 270/368 Q&A pairs...
  Uploaded 280/368 Q&A pairs...
  Uploaded 290/368 Q&A pairs...
  Uploaded 300/368 Q&A pairs...
  Uploa

## 7. Test Retrieval

In [20]:
def query_qa_pairs(query_text, top_k=5):
    """Query Weaviate for similar Q&A pairs"""
    collection = client.collections.get(collection_name)
    
    response = collection.query.near_text(
        query=query_text,
        limit=top_k,
        return_metadata=['distance']
    )
    
    print(f"\n🔍 Query: '{query_text}'")
    print(f"📊 Top {top_k} Results:\n")
    print("="*80)
    
    for i, obj in enumerate(response.objects, 1):
        distance = obj.metadata.distance
        similarity = 1 - distance
        
        print(f"\nRank {i} | Similarity: {similarity:.4f}")
        print(f"Source: {obj.properties['source']}")
        print(f"\nQ&A Pair:")
        print(obj.properties['qa_pair'][:300] + "..." if len(obj.properties['qa_pair']) > 300 else obj.properties['qa_pair'])
        print("-"*80)

# Test queries
query_qa_pairs("How do I use templates?")
query_qa_pairs("What are mockups?")


🔍 Query: 'How do I use templates?'
📊 Top 5 Results:


Rank 1 | Similarity: 0.4919
Source: intercom_customer_support

Q&A Pair:
User Question: How can I favorite or bookmark templates in Kittl, and where can I find my saved templates?
Answer: To save or bookmark a template, click the heart icon. All saved templates are automatically stored in the "Likes" section, which can be found on the left side panel of the interface.
--------------------------------------------------------------------------------

Rank 2 | Similarity: 0.4819
Source: intercom_customer_support

Q&A Pair:
User Question: Can Kittl templates be used as-is, or do they need to be modified for use?
Answer: Yes, Kittl templates can be used as-is. However, it is strongly recommended to make changes, especially for commercial use, to ensure uniqueness, help the design stand out, and reduce the risk of takedo...
--------------------------------------------------------------------------------

Rank 3 | Similarity: 0.4299
Sourc

## 8. Clean Up

In [ ]:
# Close Weaviate connection
client.close()
print("✅ Weaviate connection closed")

✅ Weaviate connection closed
